In [1]:
import optuna.visualization
import os

os.makedirs("optuna_results/plots", exist_ok=True)

In [2]:
structure_study = optuna.load_study(
    study_name="study_8p3f",
    storage="sqlite:///optuna_results/study_8p3f.db"
)

In [ ]:
fig = optuna.visualization.plot_pareto_front(
    structure_study,
    target_names=["FLOPs", "Accuracy"],
)
fig.write_image("optuna_results/plots/pareto_front_8p3f.png")
fig.show()

In [ ]:
print(f"Number of trials on the Pareto front: {len(structure_study.best_trials)}")

trial_with_highest_accuracy = max(structure_study.best_trials, key=lambda t: t.values[1])
print("Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy.number}")
print(f"\tparams: {trial_with_highest_accuracy.params}")
print(f"\tvalues: {trial_with_highest_accuracy.values}")

In [ ]:
import pandas as pd

best_trials = structure_study.best_trials

order = [
    "trial_id",
    "flops",
    "acc",
    "num_transformers",
    "embedding_dim",
    "num_heads",
    "dropout"
]

trial_dicts = []
for t in best_trials:
    entry = {
        "trial_id": t.number,
        "flops": t.values[0],
        "acc": t.values[1],
    }
    entry.update(t.params)
    trial_dicts.append(entry)

df_best = pd.DataFrame(trial_dicts)
df_best[["embedding_dim", "num_heads"]] = df_best["dim_heads"].str.split("_", expand=True).astype(int)
df_best = df_best.drop(columns=["dim_heads"])[order]
# Discard duplicates
df_best = df_best.drop_duplicates(
    subset=["flops", "acc", "num_transformers", "embedding_dim", "num_heads", "dropout"]
).reset_index(drop=True)
# Sort
df_best.sort_values(by="flops", ascending=True).reset_index(drop=True)

In [ ]:
optuna.visualization.plot_param_importances(
    structure_study, target=lambda t: t.values[0], target_name="flops"
)

In [ ]:
optuna.visualization.plot_param_importances(
    structure_study, target=lambda t: t.values[1], target_name="accuracy"
)